In [1]:
%load_ext autoreload
%autoreload 2

# Graph

## Basics/Low Level Functions

In [2]:
from kb_mcp.kb import graph

In [3]:
graph.get_node_types()

{'Concept': "Theoretical ideas (e.g., 'Standard Model')",
 'Document': 'Papers, memos, logbooks',
 'Experiment': "Experiments or runs (e.g., 'Run-2', 'Test Beam 2024')",
 'Hardware': 'Detectors, cables, chips',
 'Location': "Physical locations (e.g., 'CERN', 'Building 4')",
 'Measurement': "Specific results (e.g., '125 GeV', 'Efficiency 98%')",
 'Organization': "Institutions (e.g., 'University of Zurich', 'DOE')",
 'Person': 'Researchers, authors'}

In [4]:
graph.get_verbs()

{'authored_by': 'Authorship (Doc -> Person)',
 'cites': 'Formal citation (Doc -> Doc)',
 'influences': 'Influence relationship (Concept -> Concept, Hardware -> Hardware)',
 'located_in': 'Location relationship (Hardware -> Location)',
 'measures': 'Measurement relationship (Hardware -> Measurement / Concept)',
 'part_of': 'Component relationship (Hardware -> Hardware)',
 'produced': 'Production relationship (Experiment -> Measurement)',
 'references': 'General reference link (Doc -> Concept / Doc -> Doc)',
 'supersedes': 'Replacement relationship (Doc A -> Doc B, Concept -> Concept)'}

In [5]:
import logging
# force=True removes existing handlers so config takes effect
logging.basicConfig(level=logging.DEBUG, force=True)

node = graph.get_or_create_node("Concept", "test 130d") # type, name

DEBUG:kb_mcp.kb.graph.graph:No exact match for 'test 130d' (canonical: 'test 130d'), trying aliases and vector search
DEBUG:kb_mcp.kb.graph.graph:Found alias match for 'test 130d' in node 'test 130a'


In [6]:
node.name

'test 130a'

In [7]:
graph.add_relation("Concept", "test 0", "references", "Location", "test 130d")

DEBUG:kb_mcp.kb.graph.graph:Found exact match for 'test 0' (canonical: 'test 0')
DEBUG:kb_mcp.kb.graph.graph:Found exact match for 'test 130d' (canonical: 'test 130d')
DEBUG:kb_mcp.kb.graph.graph:Relation already exists: test 0 --[references]--> test 130d
DEBUG:kb_mcp.kb.graph.graph:Added evidence for relation 0b688073-268c-4a4a-9222-e1e5b891b969 from document None


(<GraphRelation(source=4bfab204-c5cf-42ad-b258-11a2e6956b63, verb=7ac28dd1-161e-44d8-8f98-221209d16443, target=ad2a4d80-fbed-459e-afc9-9ed6cc6e0885)>,
 False)

## Document processing

In [3]:
graph.extract_relations("Paul built the Gemini Experiment. He is the spokes person. Fish are blue.")

[]

In [4]:
from kb_mcp.kb import get

In [5]:
doc = get(limit=1, doc_type="text")

In [6]:
rel = graph.extract_relations(doc.text)
rel

[{'source_name': 'Measurement of Ab at the Z0 Resonance using Jet-Charge Technique.',
  'source_type': 'Document',
  'verb': 'authored_by',
  'target_name': 'K. Abe',
  'target_type': 'Person',
  'justification': 'K. Abe,(2) ... appears in the extensive author list of the paper.',
  'confidence': 0.96},
 {'source_name': 'Measurement of Ab at the Z0 Resonance using Jet-Charge Technique.',
  'source_type': 'Document',
  'verb': 'authored_by',
  'target_name': 'SLD Collaboration',
  'target_type': 'Organization',
  'justification': 'The header of the paper reads "The SLD Collaboration".',
  'confidence': 0.97},
 {'source_name': 'Measurement of Ab at the Z0 Resonance using Jet-Charge Technique.',
  'source_type': 'Document',
  'verb': 'references',
  'target_name': 'Department of Energy',
  'target_type': 'Organization',
  'justification': 'Work supported in part by the Department of Energy contract DE{AC03{76SF00515.',
  'confidence': 0.95},
 {'source_name': 'Measurement of Ab at the Z0 R

In [19]:
graph.process_relations(rel, document_id=doc.id)

INFO:kb_mcp.kb.graph.extraction:Processing 11 extracted relations for document 6042063d-175d-4e23-8b9c-1dfe9501c6dc
Processing relations:   0%|                                                                     | 0/11 [00:00<?, ?relation/s]DEBUG:kb_mcp.kb.graph.graph:Found exact match for 'Measurement of Ab at the Z0 Resonance using Jet-Charge Technique' (canonical: 'measurement of ab at the z0 resonance using jet-charge technique')
DEBUG:kb_mcp.kb.graph.graph:Found exact match for 'K. Abe' (canonical: 'k. abe')
DEBUG:kb_mcp.kb.graph.graph:Relation already exists: Measurement of Ab at the Z0 Resonance using Jet-Charge Technique --[authored_by]--> K. Abe
DEBUG:kb_mcp.kb.graph.graph:Added evidence for relation from document 6042063d-175d-4e23-8b9c-1dfe9501c6dc
DEBUG:kb_mcp.kb.graph.graph:Created node_map: node=761a2a2c-00cb-4bd1-8b7a-1d30dfe97328, doc=6042063d-175d-4e23-8b9c-1dfe9501c6dc
DEBUG:kb_mcp.kb.graph.graph:Updated node_map for source node 761a2a2c-00cb-4bd1-8b7a-1d30dfe97328 an

{'total': 11, 'created': 0, 'updated': 11, 'errors': 0, 'error_details': []}

### extract_and_process_document

In [7]:
doc = get(limit=1, offset=2, doc_type="text")
doc.id

'b30e68c1-5193-406b-a0ba-bb2c6d4a66fb'

In [8]:
graph.extract_and_process_document('b30e68c1-5193-406b-a0ba-bb2c6d4a66fb')

Processing relations:   0%|                                                                     | 0/12 [00:00<?, ?relation/s]/Users/scorrodi/Documents/MLAI/MCP/test-mcp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing relations: 100%|████████████████████████████████████████████████████████████| 12/12 [00:05<00:00,  2.28relation/s]


{'log_id': '38f80451-a11a-4070-8ddb-6ae82cb618fc',
 'document_id': 'b30e68c1-5193-406b-a0ba-bb2c6d4a66fb',
 'relations_extracted': 12,
 'relations_processed': 12,
 'relations_created': 11,
 'relations_updated': 1,
 'relations_errors': 0,
 'time_extraction': 46.02226710319519,
 'time_processing': 5.304998159408569,
 'extraction_model': 'openai/gpt-oss-120b',
 'hostname': 'CSI365765',
 'error_details': []}

## Get nodes related to a document

In [9]:
nodes = graph.get_nodes_for_document('b30e68c1-5193-406b-a0ba-bb2c6d4a66fb')

In [10]:
nodes[0]

{'id': '762349e1-a129-4c88-87fd-96e2480527d5',
 'name': 'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays',
 'type': 'Document',
 'aliases': [],
 'mention_count': 9,
 'node': <GraphNode(id=762349e1-a129-4c88-87fd-96e2480527d5, name=Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays, type=95969f33-65a5-412a-bf6c-37a5f7a0db91)>}

In [11]:
graph.get_document_for_node('762349e1-a129-4c88-87fd-96e2480527d5')

[{'document_id': 'b30e68c1-5193-406b-a0ba-bb2c6d4a66fb', 'mention_count': 9}]

In [12]:
node_dict = graph.get_node('762349e1-a129-4c88-87fd-96e2480527d5')

In [13]:
node_dict.keys()

dict_keys(['node', 'outgoing_relations', 'incoming_relations', 'statistics', 'linked_documents'])

In [14]:
node_dict['node']

{'id': '762349e1-a129-4c88-87fd-96e2480527d5',
 'name': 'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays',
 'type': 'Document',
 'aliases': [],
 'created_time': datetime.datetime(2025, 12, 29, 15, 11, 26, 130420, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=64800))),
 'meta': {}}

In [15]:
node_dict['statistics']

{'total_outgoing': 9,
 'total_incoming': 0,
 'total_relations': 9,
 'total_documents': 1}

In [17]:
node_dict['linked_documents']

['b30e68c1-5193-406b-a0ba-bb2c6d4a66fb']

In [16]:
node_dict['outgoing_relations']

[{'relation_id': '0e4258b9-b0e7-4d13-8fdc-8f96a0d8ef5d',
  'verb': 'references',
  'target_node': {'id': '31bcac23-a836-4626-9168-8e375f332833',
   'name': 'fragmentation models',
   'type': 'Concept'},
  'evidence_count': 1,
  'avg_confidence': 0.97,
  'created_time': datetime.datetime(2025, 12, 29, 15, 12, 5, 590300, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=64800)))},
 {'relation_id': '3069e438-148b-42ed-9c90-7b675007da68',
  'verb': 'references',
  'target_node': {'id': '213b2d41-bb24-44f4-bf37-bdf5eec86249',
   'name': 'International Europhysics Conference on High Energy Physics',
   'type': 'Document'},
  'evidence_count': 1,
  'avg_confidence': 0.94,
  'created_time': datetime.datetime(2025, 12, 29, 16, 38, 15, 754956, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=64800)))},
 {'relation_id': '36ae55a9-4abb-4a4a-8dc3-48ce08142d5d',
  'verb': 'cites',
  'target_node': {'id': 'd4ce1b6c-fe8f-4feb-923f-1a48f0019825',
   'name': 'SLD Design Report, SL

In [29]:
["'"+node_dict['node']['name'] + "' '"+a['verb'] + "' '" + a['target_node']['name']+"'" for a in node_dict['outgoing_relations']]

["'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'fragmentation models'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'International Europhysics Conference on High Energy Physics'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'cites' 'SLD Design Report, SLAC Report–273 (1984)'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'MLLA QCD'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'strangeness suppression'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'LPHD'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'references' 'XVIII International Symposium on Lepton Photon Interactions'",
 "'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays' 'authored_by' 'K. Abe'",
 "'Produ

In [24]:
node_dict['outgoing_relations'][0]['verb']

'references'

## Find Path - The Magic!

In [64]:
paths = graph.find_paths('762349e1-a129-4c88-87fd-96e2480527d5', '06449618-b7bf-479d-a908-3eb4cd0459d4')

In [65]:
paths

[{'length': 1,
  'chain': [{'element': 'node',
    'id': '762349e1-a129-4c88-87fd-96e2480527d5',
    'name': 'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays',
    'label': 'Document'},
   {'element': 'relationship',
    'id': 'd196fd1d-218c-40f6-84cd-8e89e3830525',
    'verb': 'authored_by',
    'direction': 'forward'},
   {'element': 'node',
    'id': '06449618-b7bf-479d-a908-3eb4cd0459d4',
    'name': 'SLD Collaboration',
    'label': 'Organization'}]}]

In [66]:
paths[0]['chain']

[{'element': 'node',
  'id': '762349e1-a129-4c88-87fd-96e2480527d5',
  'name': 'Production of pi+-, K+-, K0, K*0, phi, p and Lambda0 in hadronic Z0 decays',
  'label': 'Document'},
 {'element': 'relationship',
  'id': 'd196fd1d-218c-40f6-84cd-8e89e3830525',
  'verb': 'authored_by',
  'direction': 'forward'},
 {'element': 'node',
  'id': '06449618-b7bf-479d-a908-3eb4cd0459d4',
  'name': 'SLD Collaboration',
  'label': 'Organization'}]